# Aula 03 · Python e PySpark para Engenharia de Dados

## O que é
- **Python** é a língua que usamos para dar instruções nos notebooks.
- **PySpark** é o "dicionário" que traduz essas instruções para o Spark executar em paralelo. A peça central é o **DataFrame**, uma tabela com linhas e colunas.

## Por que importa
Os notebooks deste curso viram **tarefas dos jobs**. Quem escreve um bom notebook PySpark escreve uma boa tarefa.

## Como aplicar
Vamos construir uma transformação de ponta a ponta: **ler** o arquivo bruto de clientes e empréstimos, **limpar**, **agregar** e **gravar** em tabelas.

### 1. Python essencial

In [ ]:
# Variável
catalogo = "growdev"

# Lista: uma sequência de valores
tabelas = ["customers", "loans"]

# Dicionário: pares chave -> valor
caminhos = {tabela: f"/Volumes/{catalogo}/bronze/landing/{tabela}" for tabela in tabelas}

# Função: um bloco reutilizável
def nome_completo(schema, tabela):
    return f"{catalogo}.{schema}.{tabela}"  # f-string monta texto com variáveis

print(caminhos)
print(nome_completo("nivelamento", "clientes"))

### 2. API de DataFrames: read, select, filter, withColumn

In [ ]:
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.nivelamento")

df_clientes_bruto = spark.read.parquet(caminhos["customers"])  # read
display(
    df_clientes_bruto
    .select("customer_id", "full_name", "state", "declared_monthly_income")  # select
    .filter(F.col("state") == "SP")                                           # filter
    .limit(10)
)

Repare na coluna `declared_monthly_income`: tem valores como `R$ 4.905,09`, `3351.09` e `N/A`. Esse dado é **texto** e está sujo. Vamos limpar com `withColumn`.

In [ ]:
def limpar_valor(coluna):
    """'R$ 4.905,09' -> 4905.09 | '3351.09' -> 3351.09 | 'N/A' -> null"""
    texto = F.regexp_replace(F.col(coluna), "[R$\\s]", "")
    texto = F.when(texto.contains(","), F.regexp_replace(F.regexp_replace(texto, "\\.", ""), ",", ".")).otherwise(texto)
    return F.when(texto.rlike("^\\d+(\\.\\d+)?$"), texto).cast("decimal(14,2)")  # o que não é número vira null

df_clientes = (
    df_clientes_bruto
    .withColumn("renda_mensal", limpar_valor("declared_monthly_income"))
    .withColumn("segmento", F.upper(F.col("customer_segment")))
    .select("customer_id", "full_name", "state", "segmento", "renda_mensal")
    .dropDuplicates(["customer_id"])
)
display(df_clientes.limit(10))

### 3. groupBy, join e write

In [ ]:
df_emprestimos = (
    spark.read.parquet(caminhos["loans"])
    .withColumn("valor", limpar_valor("loan_amount"))
    .withColumn("status", F.initcap(F.trim(F.col("status"))))
    .withColumn("inadimplente", F.lower(F.trim(F.col("is_default"))).isin("sim", "s", "true", "1"))
    .select("loan_id", "customer_id", "loan_category", "valor", "status", "inadimplente")
    .dropDuplicates(["loan_id"])
)

df_por_estado = (
    df_emprestimos.join(df_clientes, "customer_id")       # join
    .groupBy("state")                                      # groupBy
    .agg(
        F.countDistinct("customer_id").alias("clientes"),
        F.count("loan_id").alias("emprestimos"),
        F.round(F.sum("valor"), 2).alias("valor_total"),
    )
    .orderBy(F.desc("valor_total"))
)
display(df_por_estado)

In [ ]:
df_clientes.write.mode("overwrite").saveAsTable(nome_completo("nivelamento", "clientes"))
df_emprestimos.write.mode("overwrite").saveAsTable(nome_completo("nivelamento", "emprestimos"))
df_por_estado.write.mode("overwrite").saveAsTable(nome_completo("nivelamento", "emprestimos_por_estado"))
print("Tabelas gravadas em growdev.nivelamento: clientes, emprestimos, emprestimos_por_estado")

## Checklist da aula
- [ ] Usei variáveis, listas, dicionários, funções e f-strings
- [ ] Li um arquivo bruto com `spark.read`
- [ ] Limpei colunas com `withColumn`
- [ ] Agreguei com `groupBy` e juntei tabelas com `join`
- [ ] Gravei o resultado em tabela com `saveAsTable`

> As tabelas de `growdev.nivelamento` são usadas na aula 04 (SQL).